In [2]:
%pip install --upgrade azure-cognitiveservices-speech

  Using cached azure_core-1.36.0-py3-none-any.whl.metadata (47 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
    --------------------------------------- 0.0/2.4 MB 640.0 kB/s eta 0:00:04
   -- ------------------------------------- 0.1/2.4 MB 1.2 MB/s eta 0:00:02
   ---------- ----------------------------- 0.6/2.4 MB 4.4 MB/s eta 0:00:01
   ----------------------- ---------------- 1.4/2.4 MB 7.5 MB/s eta 0:00:01
   -------------------------------------- - 2.3/2.4 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 9.1 MB/s eta 0:00:00
Using cached azure_core-1.36.0-py3-none-any.whl (213 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
from dotenv import load_dotenv
import azure.cognitiveservices.speech as speechsdk

In [ ]:
speech_key = os.getenv("CUSTOM_SPEECH_RESOURCE_KEY")
service_region = os.getenv("CUSTOM_SPEECH_REGION")
profile_id = os.getenv("CUSTOM_SPEECH_PROFILE_ID")

def speech_synthesis_to_wave_file(text: str, output_file_path: str, speaker_profile_id: str):
    # Creates an instance of a speech config with specified subscription key and service region.
    speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=service_region)
    speech_config.set_speech_synthesis_output_format(speechsdk.SpeechSynthesisOutputFormat.Riff24Khz16BitMonoPcm)
    file_config = speechsdk.audio.AudioOutputConfig(filename=output_file_path)
    speech_synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=file_config)

    # use PhoenixLatestNeural if you want word boundary event.  We will support events on DragonLatestNeural in the future.
    ssml = "<speak version='1.0' xml:lang='en-US' xmlns='http://www.w3.org/2001/10/synthesis' " \
           "xmlns:mstts='http://www.w3.org/2001/mstts'>" \
           "<voice name='DragonLatestNeural'>" \
           "<mstts:ttsembedding speakerProfileId='%s'/>" \
           "<mstts:express-as style='Prompt'>" \
           "<lang xml:lang='pt-BR'> %s </lang>" \
           "</mstts:express-as>" \
           "</voice></speak> " % (speaker_profile_id, text)

    def word_boundary(evt):
        word_info = f"Word Boundary: Text='{evt.text}', Audio offset={evt.audio_offset / 10000}ms"
        duration_info = f"Duration={evt.duration / 10000}ms, text={evt.text}"
        print(f"{word_info}, {duration_info}")

    speech_synthesizer.synthesis_word_boundary.connect(word_boundary)
    result = speech_synthesizer.speak_ssml_async(ssml).get()

    # Check result
    if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
        print("Speech synthesized for text [{}], and the audio was saved to [{}]".format(text, output_file_path))
        print("result id: {}".format(result.result_id))
    elif result.reason == speechsdk.ResultReason.Canceled:
        cancellation_details = result.cancellation_details
        print("Speech synthesis canceled: {}".format(cancellation_details.reason))
        if cancellation_details.reason == speechsdk.CancellationReason.Error:
            print("Error details: {}".format(cancellation_details.error_details))
            print("result id: {}".format(result.result_id))


In [24]:
speaker_profile_id = "Rodrigo Mendonça_20251110_16916" 
text = "Olá, este é um exemplo de síntese de fala personalizada usando o Azure Cognitive Services."
output_file_path = "output.wav"

speech_synthesis_to_wave_file(text, output_file_path, speaker_profile_id)

Speech synthesis canceled: CancellationReason.Error
Error details: Connection was closed by the remote host. Error code: 1007. Error details: `speakerProfileId or personalVoiceName` is needed in your ssml when request a PersonalVoice. USP state: TurnStarted. Received audio size: 0 bytes.
result id: ff6a06b2830d45208f1a3cb4274acfc8
